# Build Corpora with sentence-corpora

This notebook builds and pickles the stylometric corpora using the sentence-corpora package.

## Stages

1. **Setup** - Import libraries and define paths
2. **Load Data** - Read masked files from data/
3. **Build TwoLevelCorpus** - Create Sentence objects with metadata
4. **Split Corpora** - Divide into train and test sets, holding out X_operibus
5. **Save Corpora** - Pickle corpora for later use

## Stage 1: Setup

Import required libraries and define paths.

In [28]:
from __future__ import annotations

import pickle
import random
import re
from collections import defaultdict
from pathlib import Path

from sentence_corpora import Sentence, TwoLevelCorpus

## Stage 2: Load Data

Read all masked files and extract metadata from filenames.

In [30]:
DATA_DIR: Final = Path("data_processed_operibus")


def parse_filename(path: Path) -> tuple[str, str]:
    """Parse '<authorcode>_<workcode>_sentences.quesplit.masked.txt' into (author, work).

    Handles extra whitespace and underscores by splitting on '_sentences.quesplit.masked.txt'.
    """
    name = path.name
    # Remove the _sentences.quesplit.masked.txt suffix
    name = re.sub(r"_sentences\.quesplit\.masked\.txt$", "", name)
    # Split on first underscore to separate authorcode and workcode
    parts = name.split("_", 1)
    if len(parts) != 2:
        raise ValueError(
            f"Expected '<authorcode>_<workcode>' in filename, got: {path.name!r}"
        )
    return parts[0], parts[1]


def build_corpus(data_dir: Path = DATA_DIR) -> TwoLevelCorpus:
    """Read all *_sentences.quesplit.masked.txt files and return a TwoLevelCorpus."""
    sentences = []
    masked_files = sorted(data_dir.glob("*_sentences.quesplit.masked.txt"))

    for path in masked_files:
        author, work = parse_filename(path)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(
                        Sentence(
                            text=line,
                            metadata={
                                "author": author,
                                "work": work,
                            },
                        )
                    )

    return TwoLevelCorpus(sentences, levels=["author", "work"])

In [31]:
corpus = build_corpus()
print(f"Built corpus with {len(corpus)} sentences")
print(f"Authors: {corpus.get_unique_values('author')}")
unique_works = set((s.metadata["author"], s.metadata["work"]) for s in corpus)
print(f"Works: {len(unique_works)} total")

for author, work in corpus.get_unique_tuples():
    print(f"{author}_{work}")

Built corpus with 50895 sentences
Authors: ['AE', 'ASt', 'BB', 'BBi', 'BI', 'BM', 'BS', 'EM', 'H', 'HA', 'HM', 'HW', 'IS', 'M', 'NLC', 'PDE', 'PR', 'QS', 'RDV', 'RL', 'RT', 'SAC', 'SR', 'U1', 'U2', 'U3', 'U4', 'X']
Works: 34 total
AE_bhc
ASt_troilus
BB_adelae
BBi_speculum
BI_reynardus
BM_regum
BS_mathematicus
EM_mahumeti
H_carmina
H_mysterio
HA_pentateuchum
HM_gestis
HW_hortus
IS_entheticus
M_ars
M_epistulae
M_tobias
NLC_miracula
NLC_speculum
PDE_rebussiculis
PR_aurora
QS_alexandri
RDV_paulino
RL_anselmi
RT_epistulae
RT_memorabilibus
RT_miracula
SAC_excidio
SR_draco
U1_pyramo
U2_ysengrimus
U3_guiardinus
U4_liberregum
X_operibus


## Stage 3: Split Corpora

Split corpus into train and test sets, holding out the work X_operibus.

Returns (train, test, holdout) where:
  - train contains ~70% of tokens from every work except X_operibus (configurable via train_pct)
  - test contains the remaining ~30%
  - *De operibus* is held out completely (not used in train or test)

In [33]:
def train_test_split(
    corpus: TwoLevelCorpus, seed: int = 42, train_pct: float = 0.5
) -> tuple[TwoLevelCorpus, TwoLevelCorpus, TwoLevelCorpus]:
    """Split corpus into train, test, and a hold-out corpus for X_operibus.

    Args:
        corpus: The corpus to split.
        seed: Random seed for reproducibility.
        train_pct: Fraction of tokens to allocate to train (0-1). Test gets 1-train_pct.

    Returns (train, test, holdout) where:
      - holdout contains all sentences from X_operibus
      - train contains ~train_pct of tokens from every other work
      - test contains the remaining ~1-train_pct tokens
    """
    holdout_sentences = []
    test_sentences = []
    train_sentences = []

    rng = random.Random(seed)

    # Group sentences by (author, work)
    by_work = defaultdict(list)
    for s in corpus:
        key = (s.author, s.work)
        by_work[key].append(s)

    for (author, work), sentences in by_work.items():
        if work == "operibus" and author == "X":
            holdout_sentences.extend(sentences)
            continue
        # Shuffle, then greedily add sentences to train until ~train_pct of tokens reached
        shuffled = list(sentences)
        rng.shuffle(shuffled)

        total_tokens = sum(len(s.text.split()) for s in shuffled)
        target = total_tokens * train_pct

        accumulated = 0
        cutoff = len(shuffled)
        for i, s in enumerate(shuffled):
            if accumulated >= target:
                cutoff = i
                break
            accumulated += len(s.text.split())

        train_sentences.extend(shuffled[:cutoff])
        test_sentences.extend(shuffled[cutoff:])

    return (
        TwoLevelCorpus(train_sentences, levels=["author", "work"]),
        TwoLevelCorpus(test_sentences, levels=["author", "work"]),
        TwoLevelCorpus(holdout_sentences, levels=["author", "work"]),
    )

In [34]:
train_corpus, test_corpus, operibus_corpus = train_test_split(corpus, train_pct=0.7)
print(f"Train corpus: {len(train_corpus)} sentences")
print(f"Test corpus:  {len(test_corpus)} sentences")
print(f"Holdout corpus (X_operibus): {len(operibus_corpus)} sentences")
print()

Train corpus: 35474 sentences
Test corpus:  15318 sentences
Holdout corpus (X_operibus): 103 sentences



In [35]:
train_corpus.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AE              1          279     3,482        12.5
ASt             1        2,449    30,044        12.3
BB              1          486     6,567        13.5
BBi             1          677    10,012        14.8
BI              1          535     9,688        18.1
BM              1          422     5,273        12.5
BS              1          278     4,201        15.1
EM              1          373     5,764        15.5
H               2          434     5,998        13.8
HA              1          349     4,959        14.2
HM              1        2,429    28,693        11.8
HW              1        3,181    47,517        14.9
IS              1          677     9,192        13.6
M               3        1,708    22,068        12.9
NLC             2        2,511    33,420        13.3
PDE             1          599     8,239        13.8
PR              1        4,640    61,892      

In [36]:
test_corpus.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AE              1          120     1,480        12.3
ASt             1        1,048    12,871        12.3
BB              1          214     2,801        13.1
BBi             1          304     4,273        14.1
BI              1          229     4,139        18.1
BM              1          188     2,244        11.9
BS              1          125     1,800        14.4
EM              1          157     2,463        15.7
H               2          174     2,520        14.5
HA              1          149     2,119        14.2
HM              1        1,042    12,291        11.8
HW              1        1,362    20,359        14.9
IS              1          282     3,936        14
M               3          733     9,413        12.8
NLC             2        1,045    14,290        13.7
PDE             1          266     3,522        13.2
PR              1        1,995    26,521        

In [37]:
train_1000 = train_corpus.chunk(1000)
train_1000.stats()

Author      Works    Sentences    Tokens    Mean Len
--------  -------  -----------  --------  ----------
AE              3          279     3,482        12.5
ASt            29        2,449    30,044        12.3
BB              6          486     6,567        13.5
BBi             9          677    10,012        14.8
BI              9          535     9,688        18.1
BM              5          422     5,273        12.5
BS              4          278     4,201        15.1
EM              5          373     5,764        15.5
H               5          434     5,998        13.8
HA              4          349     4,959        14.2
HM             28        2,429    28,693        11.8
HW             47        3,181    47,517        14.9
IS              9          677     9,192        13.6
M              21        1,708    22,068        12.9
NLC            32        2,511    33,420        13.3
PDE             8          599     8,239        13.8
PR             61        4,640    61,892      

In [38]:
# Check authors with fewer than 5000 total tokens to make sure the splitting is working as expected
for author in train_1000.get_unique_values("author"):
    author_corpus = train_1000.filter_by_level("author", author)
    author_tokens = sum(len(s.text.split()) for s in author_corpus)
    if author_tokens < 5000:
        print(f"\n{author}: {author_tokens} tokens")
        for work_name in author_corpus.get_unique_values("work"):
            work_corpus = author_corpus.filter_by_level("work", work_name)
            work_tokens = sum(len(s.text.split()) for s in work_corpus)
            print(f"  {work_name}: {work_tokens} tokens")


AE: 3482 tokens
  bhc_1: 1157 tokens
  bhc_2: 1164 tokens
  bhc_3: 1161 tokens

BS: 4201 tokens
  mathematicus_1: 1050 tokens
  mathematicus_2: 1041 tokens
  mathematicus_3: 1057 tokens
  mathematicus_4: 1053 tokens

HA: 4959 tokens
  pentateuchum_1: 1228 tokens
  pentateuchum_2: 1239 tokens
  pentateuchum_3: 1240 tokens
  pentateuchum_4: 1252 tokens

U1: 3462 tokens
  pyramo_1: 1156 tokens
  pyramo_2: 1159 tokens
  pyramo_3: 1147 tokens

U3: 4613 tokens
  guiardinus_1: 1143 tokens
  guiardinus_2: 1150 tokens
  guiardinus_3: 1161 tokens
  guiardinus_4: 1159 tokens


## Stage 4: Save Corpora

Pickle corpora for later use.

In [40]:
with open("train_corpus.pkl", "wb") as f:
    pickle.dump(train_corpus, f)

with open("test_corpus.pkl", "wb") as f:
    pickle.dump(test_corpus, f)

with open("operibus_corpus.pkl", "wb") as f:
    pickle.dump(operibus_corpus, f)

print("Corpora pickled successfully.")
print("Files created: train_corpus.pkl, test_corpus.pkl, operibus_corpus.pkl")

Corpora pickled successfully.
Files created: train_corpus.pkl, test_corpus.pkl, operibus_corpus.pkl
